<a href="https://colab.research.google.com/github/tejalkv0/LangGraph-and-LLM/blob/main/Chain_of_thouight%2CTwo_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re, ast, torch
from typing import TypedDict, List
from transformers import AutoTokenizer, AutoModelForCausalLM
from langgraph.graph import StateGraph, END

assert torch.cuda.is_available(), (
    "No GPU found! Go to Runtime > Change runtime type > T4 GPU, then re-run from the top. "
    "On CPU this notebook takes 10+ minutes per step.")
print("GPU:", torch.cuda.get_device_name(0))

GEN_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"   # LLM 1 - writes / fixes code
REV_ID = "Qwen/Qwen2.5-3B-Instruct"           # LLM 2 - reviews code
MAX_ROUNDS = 2                                # max improve loops

def load(model_id):
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.float16, device_map="auto")
    return tok, model

generator = load(GEN_ID)
reviewer = load(REV_ID)
print("Both models loaded on:", next(generator[1].parameters()).device)

GPU: Tesla T4


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Both models loaded on: cuda:0


## 2. Helper functions

In [ ]:
def ask(llm, system, user, max_new_tokens=800, temperature=0.3):
    """Send one chat turn to a local HF model and return the text."""
    tok, model = llm
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": user}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=True, temperature=temperature, top_p=0.9)
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def extract_code(text):
    """Pull the python code block out of an LLM answer."""
    blocks = re.findall(r"```(?:python)?\n(.*?)```", text, re.S)
    return blocks[-1].strip() if blocks else text.strip()

def syntax_check(src):
    try:
        ast.parse(src)
        return True, "Syntax OK"
    except SyntaxError as e:
        return False, f"SyntaxError: {e}"

## 3. LangGraph state and nodes (chain-of-thought prompts)

In [ ]:
class State(TypedDict):
    task: str
    reasoning: str      # chain of thought from LLM 1
    code: str           # current code
    review: str         # chain of thought + feedback from LLM 2
    approved: bool
    iteration: int

GEN_SYSTEM = ("You are an expert Python programmer. First THINK STEP BY STEP: understand the task, "
              "list the inputs/outputs, plan the logic, and consider edge cases. Write this reasoning "
              "under the heading 'Reasoning:'. Then give the final complete code in one ```python block.")

REV_SYSTEM = ("You are a strict senior code reviewer. THINK STEP BY STEP: (1) does the code solve the task, "
              "(2) look for bugs and missing edge cases, (3) check readability, docstrings and efficiency. "
              "Give a short numbered list of concrete fixes. "
              "End with exactly one final line: 'VERDICT: APPROVED' if the code is good, "
              "otherwise 'VERDICT: NEEDS_FIX'.")

def draft_node(state: State):
    print("[LLM 1] Drafting code with chain of thought...")
    ans = ask(generator, GEN_SYSTEM, f"Task: {state['task']}")
    reasoning = ans.split("```")[0].strip()
    return {"reasoning": reasoning, "code": extract_code(ans), "iteration": 0}

def review_node(state: State):
    print(f"[LLM 2] Reviewing code (round {state['iteration']})...")
    ok, msg = syntax_check(state["code"])
    prompt = (f"Task: {state['task']}\n\nCode:\n```python\n{state['code']}\n```\n\n"
              f"Automatic syntax check: {msg}")
    review = ask(reviewer, REV_SYSTEM, prompt, max_new_tokens=400)
    approved = ok and "VERDICT: APPROVED" in review.upper()
    return {"review": review, "approved": approved}

def improve_node(state: State):
    print("[LLM 1] Improving code using LLM 2's review...")
    prompt = (f"Task: {state['task']}\n\nCurrent code:\n```python\n{state['code']}\n```\n\n"
              f"Reviewer feedback:\n{state['review']}\n\n"
              "Think step by step about each point of feedback, then return the full improved code "
              "in one ```python block.")
    ans = ask(generator, GEN_SYSTEM, prompt)
    return {"code": extract_code(ans), "iteration": state["iteration"] + 1}

def route(state: State):
    if state["approved"] or state["iteration"] >= MAX_ROUNDS:
        return "done"
    return "improve"

## 4. Build the LangGraph

In [ ]:
graph = StateGraph(State)
graph.add_node("draft", draft_node)
graph.add_node("review", review_node)
graph.add_node("improve", improve_node)

graph.set_entry_point("draft")
graph.add_edge("draft", "review")
graph.add_conditional_edges("review", route, {"improve": "improve", "done": END})
graph.add_edge("improve", "review")

app = graph.compile()
print("Graph ready")

Graph ready


## 5. Run it - change the task to anything you like

In [ ]:
task = "Write a Python function that checks whether a string is a valid palindrome, ignoring case, spaces and punctuation, with a few test cases."

result = app.invoke({"task": task, "reasoning": "", "code": "", "review": "",
                     "approved": False, "iteration": 0})

print("\n" + "=" * 60 + "\nLLM 1 REASONING (chain of thought)\n" + "=" * 60)
print(result["reasoning"])
print("\n" + "=" * 60 + "\nLLM 2 REVIEW\n" + "=" * 60)
print(result["review"])
print("\n" + "=" * 60 + f"\nFINAL CODE (improved rounds: {result['iteration']})\n" + "=" * 60)
print(result["code"])

[LLM 1] Drafting code with chain of thought...
[LLM 2] Reviewing code (round 0)...
[LLM 1] Improving code using LLM 2's review...
[LLM 2] Reviewing code (round 1)...
[LLM 1] Improving code using LLM 2's review...
[LLM 2] Reviewing code (round 2)...

LLM 1 REASONING (chain of thought)
Reasoning:
To solve this problem, we need to create a function that takes a string as input and returns `True` if it's a valid palindrome, considering only alphanumeric characters and ignoring case, spaces, and punctuation. Here's how we can approach this:

1. **Remove non-alphanumeric characters**: We'll use regular expressions to filter out any character that's not a letter or a digit.
2. **Convert to lowercase**: Since we're ignoring case, we'll convert all characters to lowercase.
3. **Check for palindrome property**: A string is a palindrome if it reads the same backward as forward.

Here's the step-by-step implementation:

LLM 2 REVIEW
1. The function `is_string_palindrome` should handle leading/trai